In [20]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_iris
import einops
iris = load_iris()

In [11]:
X = iris.data
y = iris.target
device = 'cuda' if torch.cuda.is_available() else 'cpu'

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)


class IrisDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


dataset = IrisDataset(X, y)
split=0.9

train_dataset = IrisDataset(X[:int(len(X)*split)], y[:int(len(X)*split)])
test_dataset = IrisDataset(X[int(len(X)*split):], y[int(len(X)*split):])

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=True
)


# example usage
for X_batch, y_batch in train_loader:
    print(X_batch)   # (batch_size, 4)
    print(y_batch)   # (batch_size,)
    break

tensor([[6.3000, 3.3000, 4.7000, 1.6000],
        [4.6000, 3.6000, 1.0000, 0.2000],
        [6.7000, 3.3000, 5.7000, 2.1000],
        [4.6000, 3.1000, 1.5000, 0.2000],
        [4.7000, 3.2000, 1.3000, 0.2000],
        [6.5000, 3.0000, 5.8000, 2.2000],
        [6.4000, 2.9000, 4.3000, 1.3000],
        [5.0000, 3.4000, 1.5000, 0.2000],
        [7.7000, 3.8000, 6.7000, 2.2000],
        [5.2000, 2.7000, 3.9000, 1.4000],
        [5.0000, 3.4000, 1.6000, 0.4000],
        [5.9000, 3.0000, 4.2000, 1.5000],
        [5.6000, 2.8000, 4.9000, 2.0000],
        [7.0000, 3.2000, 4.7000, 1.4000],
        [4.5000, 2.3000, 1.3000, 0.3000],
        [7.2000, 3.2000, 6.0000, 1.8000],
        [5.7000, 3.0000, 4.2000, 1.2000],
        [7.1000, 3.0000, 5.9000, 2.1000],
        [5.1000, 3.7000, 1.5000, 0.4000],
        [4.9000, 3.1000, 1.5000, 0.1000],
        [6.9000, 3.1000, 4.9000, 1.5000],
        [5.8000, 2.8000, 5.1000, 2.4000],
        [6.0000, 2.2000, 5.0000, 1.5000],
        [6.5000, 3.0000, 5.5000, 1

In [12]:
print(len(train_loader.dataset))
print(len(test_loader.dataset))

135
15


In [36]:
# define k means trainer class
class k_means_trainer:
    def __init__(self, 
                 k: int, 
                 train_loader: DataLoader, 
                 test_loader:DataLoader,
                 initialization_method: str = 'random')->None:
        self.k = k
        self.train_loader = train_loader
        self.test_loader = test_loader
        
        # we'll go with random initialization. We'll randomly sample k datapoints
        
        if initialization_method == 'random':
            # do random initialization
            random_idxs = torch.randint(low=0, high=len(train_loader.dataset), size=(k,))
            self.centroids, class_labels = train_loader.dataset[random_idxs]
            print(f"self.centroids = {self.centroids}")
            print(f"shape: {self.centroids.shape}")
                                        
        else:
            raise NotImplementedError()
        
    
    def fit(self, max_steps: int = 100)->tuple:
        # assign all points in self.train_loader to cluster
        # (a - b)^2 = a^2 + b^2 - 2ab
        for idx, (batch, labels) in enumerate(self.train_loader):
            asquared = einops.reduce(self.centroids**2, 'k n -> 1 k', reduction='sum') # (k, n)->(1 k)
            bsquared = einops.reduce(batch**2, 'b n -> b 1', reduction='sum') # (b, n)-> (b, 1)
            prod = 2 * einops.einsum(batch, self.centroids, 'b n, k n -> b k')
        
        
#             asquared = einops.repeat(asquared, '1 k -> b k', b=prod.shape[0])
#             bsquared = einops.repeat(bsquared, 'b 1 -> b k', k=prod.shape[1])
            
            distances = asquared + bsquared - prod # the lines above are unecessary because torch will automatically broadcast
            class_labels = torch.argmin(distances, dim=1)
            print(f"class_labels: \n{class_labels}")
            print(f"labels: \n{labels}")
            print(f"accuracy for batch {idx}: {(class_labels == labels).float().mean():.3f}")
            
            for klass in range(self.k):
                indices = class_labels[class_labels==klass]
                datapoints = batch[indices]
                centroid_new = einops.reduce(datapoints, 'b n -> 1 n', reduction='mean')
                self.centroids[klass]=centroid_new
            
            
            

        # recalcualte centroid based on cluster assignments


In [37]:
trainer = k_means_trainer(
    k=3,
    train_loader=train_loader,
    test_loader=test_loader
)

trainer.fit()

self.centroids = tensor([[4.9000, 2.5000, 4.5000, 1.7000],
        [6.0000, 2.7000, 5.1000, 1.6000],
        [5.0000, 3.2000, 1.2000, 0.2000]])
shape: torch.Size([3, 4])
class_labels: 
tensor([1, 1, 1, 2, 2, 2, 0, 1, 1, 1, 2, 0, 1, 2, 2, 1, 0, 0, 2, 1, 2, 0, 1, 0,
        0, 2, 2, 1, 2, 2, 1, 1])
labels: 
tensor([1, 2, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 2, 1,
        1, 0, 0, 2, 0, 0, 2, 2])
accuracy for batch 0: 0.250
class_labels: 
tensor([2, 0, 1, 1, 2, 0, 2, 2, 0, 1, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 0, 2, 2, 1,
        0, 0, 2, 0, 1, 0, 0, 2])
labels: 
tensor([1, 0, 2, 2, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2, 1, 0, 2, 1, 2,
        0, 0, 0, 0, 2, 0, 0, 1])
accuracy for batch 1: 0.562
class_labels: 
tensor([0, 2, 0, 0, 0, 1, 0, 1, 1, 2, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 2, 0, 1,
        1, 0, 0, 1, 0, 0, 0, 1])
labels: 
tensor([1, 2, 2, 1, 1, 0, 2, 0, 0, 2, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 2, 1, 0,
        0, 1, 1, 0, 2, 1, 2, 0])
accuracy for batch 2: 0.